# Resume Keywords Processor and Generator

This notebook executes the entire resume processing pipeline:
1. Process keywords from job descriptions
2. Update resume with high-priority keywords
3. Generate PDF output

You can customize the filenames and parameters below.

In [1]:
import os
import subprocess
import pandas as pd
import sys
import datetime
import shutil
from IPython.display import Markdown, display
import time

# Configure logging
class Logger:
    def __init__(self):
        self.logs = []
    
    def log(self, message, level="INFO"):
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = f"[{timestamp}] {level}: {message}"
        self.logs.append(log_entry)
        print(log_entry)
    
    def get_logs(self):
        return "\n".join(self.logs)

logger = Logger()

## Configuration

Set your desired filenames and parameters here:

In [2]:
# Configuration parameters
config = {
    # Input files
    "input_keywords_file": "lead_gen.csv",             # Raw keywords input
    
    # Output files
    "processed_keywords_file": "processed_keywords.csv", # Processed keywords output
    "output_resume_name": "resume",                     # Base name for output files (without extension)
    
    # Directory structure
    "markdown_dir": "markdown",                         # Directory for markdown files
    "pdf_dir": "pdf",                                   # Directory for PDF files
    
    # Options
    "top_keywords": 20,                                 # Number of top keywords to highlight
    "generate_pdf": True                                # Try to generate PDF
}

# Create directory structure
os.makedirs(config["markdown_dir"], exist_ok=True)
os.makedirs(config["pdf_dir"], exist_ok=True)

# Set file paths based on configuration
config["input_resume_file"] = os.path.join(config["markdown_dir"], f"{config['output_resume_name']}.md")
config["output_resume_file"] = config["input_resume_file"]
config["output_pdf_file"] = os.path.join(config["pdf_dir"], f"{config['output_resume_name']}.pdf")

# Check if we need to copy the existing resume into the markdown directory
if not os.path.exists(config["input_resume_file"]) and os.path.exists("resume-markdown.md"):
    logger.log(f"Copying existing resume to {config['input_resume_file']}")
    shutil.copy2("resume-markdown.md", config["input_resume_file"])

# Display configuration
for key, value in config.items():
    logger.log(f"Config: {key} = {value}")

[2025-03-20 17:15:35] INFO: Config: input_keywords_file = lead_gen.csv
[2025-03-20 17:15:35] INFO: Config: processed_keywords_file = processed_keywords.csv
[2025-03-20 17:15:35] INFO: Config: output_resume_name = resume
[2025-03-20 17:15:35] INFO: Config: markdown_dir = markdown
[2025-03-20 17:15:35] INFO: Config: pdf_dir = pdf
[2025-03-20 17:15:35] INFO: Config: top_keywords = 20
[2025-03-20 17:15:35] INFO: Config: generate_pdf = True
[2025-03-20 17:15:35] INFO: Config: input_resume_file = markdown/resume.md
[2025-03-20 17:15:35] INFO: Config: output_resume_file = markdown/resume.md
[2025-03-20 17:15:35] INFO: Config: output_pdf_file = pdf/resume.pdf


## Helper Functions

In [ ]:
def run_command(command):
    """Run a shell command and return output"""
    logger.log(f"Running command: {command}")
    try:
        result = subprocess.run(command, 
                               shell=True, 
                               check=True, 
                               text=True, 
                               capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        logger.log(f"Command failed: {e}", "ERROR")
        logger.log(f"Error output: {e.stderr}", "ERROR")
        return None

def check_file_exists(file_path):
    """Check if file exists and report its size"""
    if os.path.exists(file_path):
        size = os.path.getsize(file_path)
        logger.log(f"File exists: {file_path} (Size: {size} bytes)")
        return True
    logger.log(f"File does not exist: {file_path}", "WARNING")
    return False

def find_pandoc():
    """Find pandoc executable in various common locations"""
    pandoc_paths = [
        "pandoc",                                      # System path
        "/usr/local/bin/pandoc",                       # Standard install location
        "/opt/homebrew/bin/pandoc",                    # Homebrew on Apple Silicon
        "/opt/homebrew/Cellar/pandoc/3.6.4/bin/pandoc" # Specific Homebrew version
    ]
    
    for path in pandoc_paths:
        try:
            result = subprocess.run(f"which {path}", shell=True, capture_output=True, text=True)
            if result.returncode == 0 and result.stdout.strip():
                logger.log(f"Found pandoc at: {result.stdout.strip()}")
                return path
        except:
            continue
    
    # One more check just using which pandoc
    try:
        result = subprocess.run("which pandoc", shell=True, capture_output=True, text=True)
        if result.returncode == 0 and result.stdout.strip():
            logger.log(f"Found pandoc at: {result.stdout.strip()}")
            return "pandoc"
    except:
        pass
    
    logger.log("Pandoc not found", "WARNING")
    return None

def generate_pdf(markdown_file, output_pdf):
    """Generate PDF using pandoc or suggest alternatives"""
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(output_pdf), exist_ok=True)
    
    # Find pandoc
    pandoc_cmd = find_pandoc()
    
    if pandoc_cmd:
        try:
            # Run pandoc command with pdflatex engine explicitly specified and include the path to pdflatex
            pandoc_command = f"PATH=$PATH:/Library/TeX/texbin {pandoc_cmd} {markdown_file} -o {output_pdf} --pdf-engine=pdflatex"
            logger.log(f"Executing: {pandoc_command}")
            result = run_command(pandoc_command)
            
            if check_file_exists(output_pdf):
                logger.log(f"PDF successfully generated: {output_pdf}")
                return output_pdf
            else:
                logger.log(f"PDF was not created: {output_pdf}", "ERROR")
        except Exception as e:
            logger.log(f"PDF generation error: {e}", "ERROR")
    else:
        logger.log("Pandoc not found. You can install it with: brew install pandoc", "WARNING")
        logger.log("Alternatively, use VS Code with the Markdown PDF extension", "INFO")
    
    return None

## 1. Process Job Description Keywords

In [4]:
logger.log("Starting keyword processing...")

# Check if input file exists
if not check_file_exists(config["input_keywords_file"]):
    logger.log(f"Input keywords file not found: {config['input_keywords_file']}", "ERROR")
    raise FileNotFoundError(f"Input keywords file not found: {config['input_keywords_file']}")

# Import the keywords_processor module
sys.path.append(os.getcwd())
import keywords_processor

# Process keywords
start_time = time.time()
logger.log(f"Processing keywords from {config['input_keywords_file']}...")
high_df, low_df = keywords_processor.process_keywords(
    config["input_keywords_file"], 
    config["processed_keywords_file"]
)
end_time = time.time()
logger.log(f"Keywords processed in {end_time - start_time:.2f} seconds")

# Display top keywords
display(Markdown("### Top High-Priority Keywords"))
display(high_df.head(config["top_keywords"]))
display(Markdown("### Top Low-Priority Keywords"))
display(low_df.head(config["top_keywords"]))

[2025-03-20 17:15:35] INFO: Starting keyword processing...
[2025-03-20 17:15:35] INFO: File exists: lead_gen.csv (Size: 4979 bytes)
[2025-03-20 17:15:35] INFO: Processing keywords from lead_gen.csv...
Processed keywords saved to processed_keywords.csv
[2025-03-20 17:15:35] INFO: Keywords processed in 0.01 seconds


### Top High-Priority Keywords

,keyword,count
1,python,41
2,sql,33
44,spark,23
43,apache,21
4,data,20
36,airflow,13
0,looker,12
15,r,12
37,azure,11
41,snowflake,11


### Top Low-Priority Keywords

,keyword,count
8,data,43
13,etl,19
9,engineer,15
14,data engineer,15
17,machine learning,12
16,learning,12
15,machine,12
12,compliance,11
21,data modeling,10
18,modeling,10


## 2. Update Resume with Keywords

In [5]:
logger.log("Starting resume update...")

# Import the update_resume module
sys.path.append(os.getcwd())
import update_resume

# Load top keywords
keywords_df = pd.read_csv(config["processed_keywords_file"])
high_priority = keywords_df[(keywords_df['priority'] == 'high')].sort_values('count', ascending=False).head(config["top_keywords"])
keywords = high_priority['keyword'].tolist()

logger.log(f"Updating resume with top {len(keywords)} keywords")
for i, keyword in enumerate(keywords):
    logger.log(f"Keyword {i+1}: {keyword} (Count: {high_priority.iloc[i]['count']})")

# Update resume with keywords
start_time = time.time()
content = update_resume.highlight_keywords(
    config["input_resume_file"],
    config["output_resume_file"],
    keywords
)
end_time = time.time()
logger.log(f"Resume updated in {end_time - start_time:.2f} seconds")
logger.log(f"Updated resume saved to {config['output_resume_file']}")

[2025-03-20 17:15:35] INFO: Starting resume update...
[2025-03-20 17:15:35] INFO: Updating resume with top 20 keywords
[2025-03-20 17:15:35] INFO: Keyword 1: python (Count: 41)
[2025-03-20 17:15:35] INFO: Keyword 2: sql (Count: 33)
[2025-03-20 17:15:35] INFO: Keyword 3: spark (Count: 23)
[2025-03-20 17:15:35] INFO: Keyword 4: apache (Count: 21)
[2025-03-20 17:15:35] INFO: Keyword 5: data (Count: 20)
[2025-03-20 17:15:35] INFO: Keyword 6: airflow (Count: 13)
[2025-03-20 17:15:35] INFO: Keyword 7: looker (Count: 12)
[2025-03-20 17:15:35] INFO: Keyword 8: r (Count: 12)
[2025-03-20 17:15:35] INFO: Keyword 9: azure (Count: 11)
[2025-03-20 17:15:35] INFO: Keyword 10: snowflake (Count: 11)
[2025-03-20 17:15:35] INFO: Keyword 11: tableau (Count: 11)
[2025-03-20 17:15:35] INFO: Keyword 12: agile (Count: 10)
[2025-03-20 17:15:35] INFO: Keyword 13: databricks (Count: 9)
[2025-03-20 17:15:35] INFO: Keyword 14: bigquery (Count: 9)
[2025-03-20 17:15:35] INFO: Keyword 15: docker (Count: 8)
[2025-03-2

## 3. Generate PDF Output

In [ ]:
if config["generate_pdf"]:
    logger.log("Starting PDF generation...")
    pdf_path = generate_pdf(
        config["output_resume_file"],
        config["output_pdf_file"]
    )
    
    if pdf_path:
        logger.log(f"PDF generation complete: {pdf_path}")
        # Display a clickable link to the PDF
        from IPython.display import FileLink
        display(FileLink(pdf_path))
    else:
        logger.log("PDF generation failed. Trying direct pandoc command...", "WARNING")
        # Try direct shell command with pdflatex engine and include the path to pdflatex
        cmd = f"PATH=$PATH:/Library/TeX/texbin:/opt/homebrew/bin:/opt/homebrew/Cellar/pandoc/3.6.4/bin pandoc {config['output_resume_file']} -o {config['output_pdf_file']} --pdf-engine=pdflatex"
        logger.log(f"Executing: {cmd}")
        result = run_command(cmd)
        
        if check_file_exists(config['output_pdf_file']):
            logger.log(f"PDF successfully generated via direct command: {config['output_pdf_file']}")
            from IPython.display import FileLink
            display(FileLink(config['output_pdf_file']))
        else:
            logger.log("Direct pandoc command failed. Please use VS Code with Markdown PDF extension.", "ERROR")
            logger.log("Or install pandoc using: brew install pandoc (macOS)", "INFO")
else:
    logger.log("PDF generation skipped based on configuration")

## Summary

In [7]:
logger.log("Resume processing complete!")

# Display summary
display(Markdown("### Processing Summary"))
print(f"Input keywords file: {config['input_keywords_file']}")
print(f"Processed keywords file: {config['processed_keywords_file']}")
print(f"Resume markdown file: {config['output_resume_file']}")

if config["generate_pdf"] and os.path.exists(config['output_pdf_file']):
    print(f"Generated PDF: {config['output_pdf_file']}")

display(Markdown("### Top 10 Keywords Used"))
for i, keyword in enumerate(keywords[:10]):
    print(f"{i+1}. {keyword} (Count: {high_priority.iloc[i]['count']})")

[2025-03-20 17:15:36] INFO: Resume processing complete!


### Processing Summary

Input keywords file: lead_gen.csv
Processed keywords file: processed_keywords.csv
Resume markdown file: markdown/resume.md
Generated PDF: pdf/resume.pdf


### Top 10 Keywords Used

1. python (Count: 41)
2. sql (Count: 33)
3. spark (Count: 23)
4. apache (Count: 21)
5. data (Count: 20)
6. airflow (Count: 13)
7. looker (Count: 12)
8. r (Count: 12)
9. azure (Count: 11)
10. snowflake (Count: 11)
